# Erosion rate uncertainty calculations for CT samples
This notebook is to calculate the erosion rates unceratinties of the CT samples in order to negate any negative for subwatershed area calculations and to understand the uncertainties from Biermans paper with a 13% error propogation. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '..')

from mataquito.sample_data import load_samples, get_erosion_rates, get_areas
from mataquito.flow_network import all_subwatershed_erosion_rates, HEADWATERS

Subcatchment erosion rates for CT-6,5,4,10,11,8,2,9

In [ ]:
# Load sample data from the canonical Excel file
df = load_samples()
E = get_erosion_rates(df)
A = get_areas(df)

print("Loaded from MataquitoSampleData.xlsx:")
for sid in sorted(E.keys(), key=lambda x: int(x.split('-')[1])):
    print(f"  {sid}: E = {E[sid]:.1f} m/Myr, A = {A[sid]:.2f} km²")

In [ ]:
# Compute all subwatershed erosion rates using mass balance
# Flow network: CT-7→CT-5, CT-1→CT-6, CT-5+CT-6→CT-10→CT-11→CT-8→CT-2, CT-2+CT-3→CT-9
# Note: CT-4 is skipped (problematic data), lumped into CT-10 calculation

E_sub = all_subwatershed_erosion_rates(E, A)

print("=" * 70)
print("SUBWATERSHED EROSION RATES (m/Myr)")
print("=" * 70)
print("\nHeadwater subcatchments (original values):")
for hw in HEADWATERS:
    print(f"  {hw}: {E_sub[hw]:.2f}")
print("\nCalculated subwatersheds:")
for site in ['CT-5', 'CT-6', 'CT-10', 'CT-11', 'CT-8', 'CT-2', 'CT-9']:
    print(f"  {site}: {E_sub[site]:.2f}")
print(f"\n  CT-4: {E['CT-4']:.2f} (original - not used in hierarchy)")
print("\nNote: Negative values indicate net sediment storage/deposition")
print("=" * 70)

In [ ]:
# Detailed subwatershed calculation with flux breakdown
from mataquito.flow_network import FLOW_NETWORK, TRAVERSAL_ORDER, subwatershed_area

print("=" * 70)
print("DETAILED SUBWATERSHED EROSION RATE CALCULATIONS")
print("=" * 70)

for site in TRAVERSAL_ORDER:
    parents = FLOW_NETWORK[site]
    A_sub = subwatershed_area(site, A)
    upstream_flux = sum(E[p] * A[p] for p in parents)
    site_flux = E[site] * A[site]
    print(f"\n{'-'*70}")
    print(f"{site} (receives from {' + '.join(parents)}):")
    for p in parents:
        print(f"  Flux from {p}: {E[p] * A[p]:.2f}")
    if len(parents) > 1:
        print(f"  Total upstream flux: {upstream_flux:.2f}")
    print(f"  Flux at {site}: {site_flux:.2f}")
    print(f"  Subwatershed area: {A_sub:.2f} km²")
    print(f"  E_sub = {E_sub[site]:.2f} M/myr")

### CT-5 Subwatershed Area

In [7]:
# CT-5 subwatershed (receives from CT-7)
Ea = E['CT-5']  # Downstream erosion rate
Eb = E['CT-7']  # Upstream erosion rate
Aa = A['CT-5']  # Downstream area
Ab = A['CT-7']  # Upstream area
Asub = Aa - Ab  # Subwatershed area

# Calculate using Ēsub = (ĒdownstreamAdownstream - ĒupstreamAupstream) / Asub
Ec_CT5 = (Ea * Aa - Eb * Ab) / Asub

print(f"CT-5 subwatershed erosion rate: {Ec_CT5:.2f}")
print(f"Downstream (CT-5): {Ea:.2f}")
print(f"Upstream (CT-7): {Eb:.2f}")
print(f"Downstream area: {Aa:.2f}")
print(f"Upstream area: {Ab:.2f}")
print(f"Subwatershed area: {Asub:.2f}")

CT-5 subwatershed erosion rate: 152.21
Downstream (CT-5): 413.27
Upstream (CT-7): 475.41
Downstream area: 1495.65
Upstream area: 1208.09
Subwatershed area: 287.57


### CT-6 Subwatershed area

In [5]:
# CT-6 subwatershed (receives from CT-1)
Ea = E['CT-6']  # Downstream erosion rate
Eb = E['CT-1']  # Upstream erosion rate
Aa = A['CT-6']  # Downstream area
Ab = A['CT-1']  # Upstream area
Asub = Aa - Ab  # Subwatershed area

# Calculate using Ēsub = (ĒdownstreamAdownstream - ĒupstreamAupstream) / Asub
Ec_CT6 = (Ea * Aa - Eb * Ab) / Asub

print(f"CT-6 subwatershed erosion rate: {Ec_CT6:.2f}")
print(f"Downstream (CT-6): {Ea:.2f}")
print(f"Upstream (CT-1): {Eb:.2f}")
print(f"Downstream area: {Aa:.2f}")
print(f"Upstream area: {Ab:.2f}")
print(f"Subwatershed area: {Asub:.2f}")

CT-6 subwatershed erosion rate: 37.65
Downstream (CT-6): 29.79
Upstream (CT-1): 22.90
Downstream area: 2595.90
Upstream area: 1382.40
Subwatershed area: 1213.51

Calculation: (29.79 × 2595.90 - 22.90 × 1382.40) ÷ 1213.51
